In [1]:
# TODO : tester retrieval_task et generation_task separement

In [2]:
import os
from datetime import datetime
from langfuse import get_client
from langchain_qdrant import QdrantVectorStore, RetrievalMode

from polylex_chatbot.env import load_project_env

env_path = load_project_env()

from experiment.tasks import make_rag_task
from experiment.evaluators import *

from polylex_chatbot.config import EMBEDDING_MODEL_CONFIG, get_sparse_model_config_fr, DB_DENSE_INDEX_CONFIG, DB_SPARSE_INDEX_CONFIG_FR, LLM_MODEL_CONFIG, PROMPT_TEMPLATE_FR

In [3]:
langfuse = get_client()
DATASET_NAME = "20250520_clean_dev_dataset"
dataset = langfuse.get_dataset(DATASET_NAME)
RUN_NAME = datetime.now().isoformat()

embeddings = EMBEDDING_MODEL_CONFIG

qdrant = QdrantVectorStore.from_existing_collection(
    url=os.getenv("QDRANT_URL"),
    embedding=embeddings,
    sparse_embedding=get_sparse_model_config_fr(),
    collection_name=os.getenv("DB_COLLECTION_NAME"),
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name=list(DB_DENSE_INDEX_CONFIG.keys())[0],
    sparse_vector_name=list(DB_SPARSE_INDEX_CONFIG_FR.keys())[0]
)

NB_CHUNKS_RETRIEVED = 100
NB_CHUNKS_SEND_TO_LLM = 5

In [4]:
rag_result = dataset.run_experiment(
    name=os.getenv("DB_COLLECTION_NAME"),
    run_name=f"{RUN_NAME}-rag",
    description=f"Add semantic similarity metric on {DATASET_NAME} dataset using {os.getenv("DB_COLLECTION_NAME")} collection",
    task=make_rag_task(qdrant, os.getenv("MODEL_RERANKER_NAME"), os.getenv("MODEL_RERANKER_API_KEY"), os.getenv("MODELS_BASE_URL"), NB_CHUNKS_RETRIEVED, LLM_MODEL_CONFIG, NB_CHUNKS_SEND_TO_LLM, PROMPT_TEMPLATE_FR),
    evaluators=[
        # retrieval
        make_hit_at_x_evaluator(1),
        make_hit_at_x_evaluator(2),
        make_hit_at_x_evaluator(3),
        make_hit_at_x_evaluator(4),
        make_hit_at_x_evaluator(5),
        make_hit_at_x_evaluator(10),
        make_hit_at_x_evaluator(15),
        make_hit_at_x_evaluator(20),
        mrr_doc_evaluator,
        nb_correct_doc_evaluator,
        # generation
        chrf_evaluator,
        len_ratio_answers_evaluator,
        semantic_similarity_evaluator
    ],
    metadata={
        "collection_name": os.getenv("DB_COLLECTION_NAME"),
        "top_k": NB_CHUNKS_RETRIEVED
    }
)

print(rag_result.format())

Task exception was never retrieved
future: <Task finished name='Task-62' coro=<AsyncClient.aclose() done, defined at /home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/httpx/_client.py:1978> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/httpx/_client.py", line 1985, in aclose
    await self._transport.aclose()
  File "/home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 406, in aclose
    await self._pool.aclose()
  File "/home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 353, in aclose
    await self._close_connections(closing_connections)
  File "/home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 345, in _close_connections
    await connection.aclose()
  File "/home/saskya/de

Individual Results: Hidden (13 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: 20260616_164451_collection
📋 Run name: 2026-06-21T14:34:13.196581-rag - Add semantic similarity metric on 20250520_clean_dev_dataset dataset using 20260616_164451_collection collection\n13 items\nEvaluations:\n  • hit_at_3\n  • hit_at_10\n  • hit_at_2\n  • hit_at_20\n  • len_ratio_answers\n  • semantic_similarity\n  • nb_correct_doc\n  • hit_at_1\n  • hit_at_15\n  • hit_at_4\n  • mrr_doc\n  • chrf_score\n  • hit_at_5\n\nAverage Scores:\n  • hit_at_3: 0.615\n  • hit_at_10: 0.769\n  • hit_at_2: 0.615\n  • hit_at_20: 0.769\n  • len_ratio_answers: 0.462\n  • nb_correct_doc: 2.923\n  • hit_at_1: 0.615\n  • hit_at_15: 0.769\n  • hit_at_4: 0.692\n  • mrr_doc: 0.646\n  • hit_at_5: 0.692\n\n🔗 Dataset Run:\n   http://localhost:3000/project/cmmvqmjqn0007nw07hbqsche1/datasets/cmpdor8yr0026nw07btz59oao/runs/3e759c50-9656-4874-ab67-3663f5218928
